In [ ]:
import pandas as pd
import os
import re

# Set your Downloads folder path
downloads_folder = os.path.join(os.path.expanduser("~"), "Downloads")
input_file_path = os.path.join(downloads_folder, "Scored Lineage Model.xlsx")
output_file_path = os.path.join(downloads_folder, "Lineage_GUIDs_With_Duplicates.xlsx")

# Read the Excel file (first sheet only)
df = pd.read_excel(input_file_path, sheet_name=0, engine="openpyxl")

# Flatten all cell values into a single list of strings
all_cells = df.astype(str).values.flatten().tolist()

# Regex pattern to find lineageTag and sourceLineageTag with GUIDs
pattern = r"\b(lineageTag|sourceLineageTag):\s*([a-f0-9\-]{36})"

# Extract matches
matches = [re.search(pattern, cell) for cell in all_cells if re.search(pattern, cell)]

# Create DataFrame with tag type and GUID
data = [(match.group(1), match.group(2)) for match in matches]
tags_df = pd.DataFrame(data, columns=["Tag Type", "GUID"])

# Identify duplicates (keep all rows with duplicate GUIDs)
duplicate_df = tags_df[tags_df.duplicated("GUID", keep=False)]

# Write to Excel with two sheets
with pd.ExcelWriter(output_file_path, engine="openpyxl") as writer:
    tags_df.to_excel(writer, sheet_name="All GUIDs", index=False)
    duplicate_df.to_excel(writer, sheet_name="Duplicate GUIDs", index=False)

print(f"Processed file saved to: {output_file_path}")
